In [43]:
from ssfaitk.utils.trip_file_loader import load_random_trips, list_available_trips, load_trips
from ssfaitk.viz.interactive_maps import create_heatmap_html

# Load sample of fleet
# df = load_random_trips(1000)  # 100 random trips
trips = list_available_trips()


Found 7862 trips in /Users/altarturhamza/Documents/GitHub/ssf-ai-toolkit/examples/data/pds-trips/zanzibar


In [44]:
df = load_trips(trips[:1000])

from ssfaitk.models.effort.statistical_effort_enhanced import StatisticalEffortClassifier
from ssfaitk.models.effort.statistical_effort import StatisticalEffortClassifier as SEC


clf = SEC()

# One-line prediction with default settings
predictions_simple = clf.predict(df)

# Create combined heatmap
create_heatmap_html(
    predictions_simple,
    output_path='fleet_heatmap - 2500.html',
    radius=15,
    blur=25
)

print(f"Created heatmap from {predictions_simple['trip_id'].nunique()} trips")

Loading 1000 trips...
Loading trip 12067898 from /Users/altarturhamza/Documents/GitHub/ssf-ai-toolkit/examples/data/pds-trips/zanzibar/pds-tracks_12067898.parquet
Loaded 1,343 points for trip 12067898
  [1/1000] Trip 12067898: 1,343 points ✓
Loading trip 12067900 from /Users/altarturhamza/Documents/GitHub/ssf-ai-toolkit/examples/data/pds-trips/zanzibar/pds-tracks_12067900.parquet
Loaded 2,864 points for trip 12067900
  [2/1000] Trip 12067900: 2,864 points ✓
Loading trip 12068874 from /Users/altarturhamza/Documents/GitHub/ssf-ai-toolkit/examples/data/pds-trips/zanzibar/pds-tracks_12068874.parquet
Loaded 2,017 points for trip 12068874
  [3/1000] Trip 12068874: 2,017 points ✓
Loading trip 12071670 from /Users/altarturhamza/Documents/GitHub/ssf-ai-toolkit/examples/data/pds-trips/zanzibar/pds-tracks_12071670.parquet
Loaded 2,522 points for trip 12071670
  [4/1000] Trip 12071670: 2,522 points ✓
Loading trip 12072167 from /Users/altarturhamza/Documents/GitHub/ssf-ai-toolkit/examples/data/pds-

Created heatmap from 1000 trips


In [56]:
from ssfaitk.utils.shore_distance_filter_updated import add_shore_filtering
df_filtered = add_shore_filtering(
    predictions_simple,
    region='zanzibar',
    method='coastline',
    coastline_shapefile='../../data/coastline/lines.shp',
    land_shapefile='../../data/coastline/GSHHS_f_L1.shp',
    min_distance_km=0.5,
    filter_land_points=True
)

print("✓ Filtering complete!")
print(f"  Near-shore removed: {df_filtered['is_near_shore'].sum():,}")
print(f"  On-land removed: {df_filtered['is_on_land'].sum():,}")

Region 'zanzibar' bbox: (-7.0, -4.5, 38.5, 40.5)
Loading: ../../data/coastline/lines.shp
⚡ Cropped: 858199 → 210 features (100.0% reduction)
✓ Coastline: 210 segments
Loading land: ../../data/coastline/GSHHS_f_L1.shp
✓ Land: 117 polygons
Filter mode: Only processing 682,802 fishing points (skipping 2,032,251 non-fishing)
Computing distances for 682,802 points...
  10,000/682,802 (1.5%)
  20,000/682,802 (2.9%)
  30,000/682,802 (4.4%)
  40,000/682,802 (5.9%)
  50,000/682,802 (7.3%)
  60,000/682,802 (8.8%)
  70,000/682,802 (10.3%)
  80,000/682,802 (11.7%)
  90,000/682,802 (13.2%)
  100,000/682,802 (14.6%)
  110,000/682,802 (16.1%)
  120,000/682,802 (17.6%)
  130,000/682,802 (19.0%)
  140,000/682,802 (20.5%)
  150,000/682,802 (22.0%)
  160,000/682,802 (23.4%)
  170,000/682,802 (24.9%)
  180,000/682,802 (26.4%)
  190,000/682,802 (27.8%)
  200,000/682,802 (29.3%)
  210,000/682,802 (30.8%)
  220,000/682,802 (32.2%)
  230,000/682,802 (33.7%)
  240,000/682,802 (35.1%)
  250,000/682,802 (36.6%)


✓ Filtering complete!
  Near-shore removed: 183,845
  On-land removed: 103,103


In [46]:
import pydeck as pdk
import h3

fishing = predictions_simple[predictions_simple['is_fishing'] == 1]

# Aggregate
fishing['h3'] = [
    h3.latlng_to_cell(lat, lon, 8)
    for lat, lon in zip(fishing['latitude'], fishing['longitude'])
]

hex_counts = fishing.groupby('h3').size().reset_index(name='count')
hex_counts['lat'] = hex_counts['h3'].apply(lambda x: h3.cell_to_latlng(x)[0])
hex_counts['lon'] = hex_counts['h3'].apply(lambda x: h3.cell_to_latlng(x)[1])

# Normalize counts for color
max_count = hex_counts['count'].max()
hex_counts['color_r'] = (hex_counts['count'] / max_count * 255).astype(int)
hex_counts['color_g'] = 255 - hex_counts['color_r']

# 3D Column layer
layer = pdk.Layer(
    'ColumnLayer',
    data=hex_counts,
    get_position=['lon', 'lat'],
    get_elevation='count',
    elevation_scale=2,
    radius=300,
    get_fill_color='[color_r, color_g, 255, 180]',
    pickable=True,
    auto_highlight=True
)

view_state = pdk.ViewState(
    latitude=hex_counts['lat'].mean(),
    longitude=hex_counts['lon'].mean(),
    zoom=10,
    pitch=50,
    bearing=0
)

r = pdk.Deck(
    layers=[layer],
    initial_view_state=view_state,
    map_style='https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json'
)

r.to_html('fishing_3d_1000_trips.html')

/var/folders/kj/2hjk7hzn2qz7f8rsq4mzqnm40000gp/T/ipykernel_10800/773600524.py:7: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [57]:
import pydeck as pdk
import h3


df_filtered_ = df_filtered[df_filtered['is_near_shore']!= 1]
df_filtered_ = df_filtered_[~df_filtered_['is_on_land']]

fishing = df_filtered_[df_filtered_['is_fishing'] == 1]


print(f"  Near-shore still: {df_filtered_['is_near_shore'].sum():,}")
print(f"  On-land still: {df_filtered_['is_on_land'].sum():,}")
# Aggregate
fishing['h3'] = [
    h3.latlng_to_cell(lat, lon, 8)
    for lat, lon in zip(fishing['latitude'], fishing['longitude'])
]

hex_counts = fishing.groupby('h3').size().reset_index(name='count')
hex_counts['lat'] = hex_counts['h3'].apply(lambda x: h3.cell_to_latlng(x)[0])
hex_counts['lon'] = hex_counts['h3'].apply(lambda x: h3.cell_to_latlng(x)[1])

# Normalize counts for color
max_count = hex_counts['count'].max()
hex_counts['color_r'] = (hex_counts['count'] / max_count * 255).astype(int)
hex_counts['color_g'] = 255 - hex_counts['color_r']

# 3D Column layer
layer = pdk.Layer(
    'ColumnLayer',
    data=hex_counts,
    get_position=['lon', 'lat'],
    get_elevation='count',
    elevation_scale=2,
    radius=300,
    get_fill_color='[color_r, color_g, 255, 180]',
    pickable=True,
    auto_highlight=True
)

view_state = pdk.ViewState(
    latitude=hex_counts['lat'].mean(),
    longitude=hex_counts['lon'].mean(),
    zoom=10,
    pitch=50,
    bearing=0
)

r = pdk.Deck(
    layers=[layer],
    initial_view_state=view_state,
    map_style='https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json'
)

r.to_html('fishing_3d_1000_trips_filtered_0.5km.html')

  Near-shore still: 0
  On-land still: 0


/var/folders/kj/2hjk7hzn2qz7f8rsq4mzqnm40000gp/T/ipykernel_10800/2207725647.py:14: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [2]:
df = predictions_simple

In [18]:
import numpy as np
import pandas as pd

df = df_fish.copy()
df["weight"] = df["is_fishing"].astype(float)  # or probability

# Choose grid size (in degrees). 0.03 ≈ 3.3 km at equator – tweak for your case.
CELL_SIZE = 0.03

df["lon_bin"] = (np.floor(df["longitude"] / CELL_SIZE) * CELL_SIZE) + CELL_SIZE / 2
df["lat_bin"] = (np.floor(df["latitude"] / CELL_SIZE) * CELL_SIZE) + CELL_SIZE / 2

agg = (
    df.groupby(["lon_bin", "lat_bin"], as_index=False)
      .agg({"weight": "sum"})
      .rename(columns={"lon_bin": "lon", "lat_bin": "lat"})
)

print(len(df), "original points →", len(agg), "grid cells")

import pydeck as pdk

CENTER_LAT, CENTER_LON = -8.6, 126.0

view_state = pdk.ViewState(
    latitude=CENTER_LAT,
    longitude=CENTER_LON,
    zoom=7,
    pitch=55,
    bearing=0,
)

col_layer = pdk.Layer(
    "ColumnLayer",
    agg,
    get_position='[lon, lat]',
    get_elevation="weight",
    elevation_scale=30,      # tune vertical exaggeration
    radius=1500,             # column radius in meters
    extruded=True,
    pickable=True,
    auto_highlight=True,
)

tooltip = {
    "html": "<b>Fishing intensity:</b> {weight}",
    "style": {"backgroundColor": "black", "color": "white"}
}

r = pdk.Deck(
    layers=[col_layer],
    initial_view_state=view_state,
    tooltip=tooltip,
    map_style="mapbox://styles/mapbox/dark-v10",
)

r.to_html("timor_fishing_grid.html", notebook_display=False)


265518 original points → 423 grid cells


In [24]:
import pydeck as pdk
import pandas as pd

fishing = df[df['is_fishing'] == 1]

# Create layer with 3D hexagons
layer = pdk.Layer(
    'GridLayer',
    data=fishing,
    get_position=['longitude', 'latitude'],
    cell_size=1000,  # Grid cell size
    elevation_scale=50,
    extruded=True,
    pickable=True,
    colorRange=[
        [1, 152, 189],
        [73, 227, 206],
        [216, 254, 181],
        [254, 237, 177],
        [254, 173, 84],
        [209, 55, 78]
    ]
)

# Set initial view
view_state = pdk.ViewState(
    latitude=fishing['latitude'].mean(),
    longitude=fishing['longitude'].mean(),
    zoom=10,
    pitch=45,  # Tilt angle (0=top-down, 60=angled)
    bearing=0
)

# Create deck
r = pdk.Deck(
    layers=[layer],
    initial_view_state=view_state,
    map_style='mapbox://styles/mapbox/dark-v10',  # Dark map like your image
    tooltip={'text': 'Fishing Points: {elevationValue}'}
)

r.to_html('fishing_3d_hexagon_map.html')
print("Saved to fishing_3d_hexagon_map.html")

Saved to fishing_3d_hexagon_map.html


In [25]:
import pydeck as pdk
import h3
import pandas as pd

fishing = df[df['is_fishing'] == 1]

# Aggregate to H3 hexagons FIRST
fishing['h3'] = [
    h3.latlng_to_cell(lat, lon, 8)  # resolution 8 = ~500m
    for lat, lon in zip(fishing['latitude'], fishing['longitude'])
]

# Count per hexagon
hex_agg = fishing.groupby('h3').size().reset_index(name='count')

# Get hex centers
hex_agg['lat'] = hex_agg['h3'].apply(lambda x: h3.cell_to_latlng(x)[0])
hex_agg['lon'] = hex_agg['h3'].apply(lambda x: h3.cell_to_latlng(x)[1])

print(f"Reduced from {len(fishing):,} points to {len(hex_agg):,} hexagons")

# Now visualize (much smaller!)
layer = pdk.Layer(
    'HexagonLayer',
    data=hex_agg,
    get_position=['lon', 'lat'],
    get_weight='count',  # Use pre-aggregated counts
    radius=500,
    elevation_scale=50,
    extruded=True,
    pickable=True,
    colorRange=[
        [1, 152, 189],
        [73, 227, 206],
        [216, 254, 181],
        [254, 237, 177],
        [254, 173, 84],
        [209, 55, 78]
    ]
)

view_state = pdk.ViewState(
    latitude=fishing['latitude'].mean(),
    longitude=fishing['longitude'].mean(),
    zoom=10,
    pitch=45,
    bearing=0
)

r = pdk.Deck(
    layers=[layer],
    initial_view_state=view_state,
    map_style='mapbox://styles/mapbox/dark-v10',
    tooltip={'text': 'Fishing Points: {count}'}
)

r.to_html('fishing_3d_small.html')
print(f"HTML size: ~{len(hex_agg) * 0.001:.1f} MB")

Reduced from 265,518 points to 1,494 hexagons
HTML size: ~1.5 MB


In [26]:
import folium
from folium.plugins import HeatMap
import h3

fishing = df[df['is_fishing'] == 1]

# Aggregate to hexagons
fishing['h3'] = [
    h3.latlng_to_cell(lat, lon, 8)
    for lat, lon in zip(fishing['latitude'], fishing['longitude'])
]

hex_counts = fishing.groupby('h3').size().reset_index(name='count')
hex_counts['geometry'] = [
    [[lat, lon] for lat, lon in h3.cell_to_boundary(h)]
    for h in hex_counts['h3']
]

# Create map with dark background
m = folium.Map(
    location=[fishing['latitude'].mean(), fishing['longitude'].mean()],
    zoom_start=10,
    tiles='CartoDB dark_matter'
)

# Color scale
max_count = hex_counts['count'].max()

# Add 3D-looking hexagons with gradient
for _, row in hex_counts.iterrows():
    intensity = row['count'] / max_count
    
    # Color gradient: blue -> cyan -> yellow -> red
    if intensity < 0.25:
        color = f'rgb(1, {int(152 + intensity*400)}, 189)'  # Blue to cyan
    elif intensity < 0.5:
        color = f'rgb({int((intensity-0.25)*800)}, 227, {int(206 - (intensity-0.25)*400)})'  # Cyan to green
    elif intensity < 0.75:
        color = f'rgb(254, {int(237 - (intensity-0.5)*400)}, {int(177 - (intensity-0.5)*400)})'  # Yellow
    else:
        color = f'rgb({int(254 - (intensity-0.75)*180)}, {int(173 - (intensity-0.75)*470)}, 84)'  # Orange to red
    
    folium.Polygon(
        locations=row['geometry'],
        color=color,
        fill=True,
        fillColor=color,
        fillOpacity=0.7,
        weight=2,
        popup=f"Fishing: {row['count']}"
    ).add_to(m)

m.save('fishing_heatmap_with_map.html')
print("Saved with map background!")

Saved with map background!


In [27]:
import pydeck as pdk
import h3

fishing = df[df['is_fishing'] == 1]

# Aggregate
fishing['h3'] = [
    h3.latlng_to_cell(lat, lon, 8)
    for lat, lon in zip(fishing['latitude'], fishing['longitude'])
]

hex_counts = fishing.groupby('h3').size().reset_index(name='count')
hex_counts['lat'] = hex_counts['h3'].apply(lambda x: h3.cell_to_latlng(x)[0])
hex_counts['lon'] = hex_counts['h3'].apply(lambda x: h3.cell_to_latlng(x)[1])

# 3D Hexagon layer
layer = pdk.Layer(
    'HexagonLayer',
    data=hex_counts,
    get_position=['lon', 'lat'],
    get_weight='count',
    radius=500,
    elevation_scale=50,
    elevation_range=[0, 3000],
    extruded=True,  # THIS makes it 3D!
    pickable=True,
    auto_highlight=True,
    colorRange=[
        [1, 152, 189],
        [73, 227, 206],
        [216, 254, 181],
        [254, 237, 177],
        [254, 173, 84],
        [209, 55, 78]
    ]
)

view_state = pdk.ViewState(
    latitude=hex_counts['lat'].mean(),
    longitude=hex_counts['lon'].mean(),
    zoom=10,
    pitch=50,  # Tilt for 3D view!
    bearing=0
)

# Use Carto basemap (FREE, no token!)
r = pdk.Deck(
    layers=[layer],
    initial_view_state=view_state,
    map_style='https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json',  # FREE!
)

r.to_html('fishing_3d_carto.html')
print("3D hexagons with free Carto basemap!")

3D hexagons with free Carto basemap!


In [32]:
import pydeck as pdk
import h3

fishing = df[df['is_fishing'] == 1]

# Aggregate
fishing['h3'] = [
    h3.latlng_to_cell(lat, lon, 8)
    for lat, lon in zip(fishing['latitude'], fishing['longitude'])
]

hex_counts = fishing.groupby('h3').size().reset_index(name='count')
hex_counts['lat'] = hex_counts['h3'].apply(lambda x: h3.cell_to_latlng(x)[0])
hex_counts['lon'] = hex_counts['h3'].apply(lambda x: h3.cell_to_latlng(x)[1])

# Normalize counts for color
max_count = hex_counts['count'].max()
hex_counts['color_r'] = (hex_counts['count'] / max_count * 255).astype(int)
hex_counts['color_g'] = 255 - hex_counts['color_r']

# 3D Column layer
layer = pdk.Layer(
    'ColumnLayer',
    data=hex_counts,
    get_position=['lon', 'lat'],
    get_elevation='count',
    elevation_scale=2,
    radius=300,
    get_fill_color='[color_r, color_g, 255, 180]',
    pickable=True,
    auto_highlight=True
)

view_state = pdk.ViewState(
    latitude=hex_counts['lat'].mean(),
    longitude=hex_counts['lon'].mean(),
    zoom=10,
    pitch=50,
    bearing=0
)

r = pdk.Deck(
    layers=[layer],
    initial_view_state=view_state,
    map_style='https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json'
)

r.to_html('fishing_3d_columns.html')

In [33]:
from ssfaitk.utils.shore_distance_filter_updated import add_shore_filtering
df_filtered = add_shore_filtering(
    df,
    region='zanzibar',
    method='coastline',
    coastline_shapefile='../../data/coastline/lines.shp',
    land_shapefile='../../data/coastline/GSHHS_f_L1.shp',
    min_distance_km=0.5,
    filter_land_points=True
)

print("✓ Filtering complete!")
print(f"  Near-shore removed: {df_filtered['is_near_shore'].sum():,}")
print(f"  On-land removed: {df_filtered['is_on_land'].sum():,}")

Region 'zanzibar' bbox: (-7.0, -4.5, 38.5, 40.5)
Loading: ../../data/coastline/lines.shp
⚡ Cropped: 858199 → 210 features (100.0% reduction)
✓ Coastline: 210 segments
Loading land: ../../data/coastline/GSHHS_f_L1.shp
✓ Land: 117 polygons
Filter mode: Only processing 265,518 fishing points (skipping 0 non-fishing)
Computing distances for 265,518 points...
  10,000/265,518 (3.8%)
  20,000/265,518 (7.5%)
  30,000/265,518 (11.3%)
  40,000/265,518 (15.1%)
  50,000/265,518 (18.8%)
  60,000/265,518 (22.6%)
  70,000/265,518 (26.4%)
  80,000/265,518 (30.1%)
  90,000/265,518 (33.9%)
  100,000/265,518 (37.7%)
  110,000/265,518 (41.4%)
  120,000/265,518 (45.2%)
  130,000/265,518 (49.0%)
  140,000/265,518 (52.7%)
  150,000/265,518 (56.5%)
  160,000/265,518 (60.3%)
  170,000/265,518 (64.0%)
  180,000/265,518 (67.8%)
  190,000/265,518 (71.6%)
  200,000/265,518 (75.3%)
  210,000/265,518 (79.1%)
  220,000/265,518 (82.9%)
  230,000/265,518 (86.6%)
  240,000/265,518 (90.4%)
  250,000/265,518 (94.2%)
  26

✓ Filtering complete!
  Near-shore removed: 65,033
  On-land removed: 26,910
